# **🏋🏽 Wearable Biomechanics - Annotation Check**

### 🔧 Colab Setup & Library Init

In [5]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

# 1. Colab-specific: mount Google Drive & configure renderers
try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

# 2. Ensure dependencies are installed
if importlib.util.find_spec("open_dataset_store") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/janithcyapa/OpenDatasetStore.git"])
for pkg in ("cv2", "ipywidgets"):
    mod = "opencv-python" if pkg == "cv2" else pkg
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", mod])

# 3. Clone (or pull) public repo into /content/Wearable-Biomechanics and add to sys.path
repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "cleannjerk_dataset" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import init_store, display_table

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}
store = init_store(dataset_path=dataset_path)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Store initialised at: /content/drive/MyDrive/Datasets/Wearable-Biomechanics (Backend: local)


### 📊 Dataset & Trials Overview

In [ ]:
# @title 📊 Dataset & Trials Overview
store.summary()
display_table(store.list_entries(entry_type="complete_trials", include_paths=False))
display_table(store.list_entries(entry_type="fault_tolerant_trials", include_paths=True))

### 📝 Interactive Annotation Editor & Saver

In [ ]:
# @title 📝 Interactive Annotation Editor & Saver { display-mode: "form" }
from src.annotation import launch_annotation_checker

ENTRY_TYPE = "fault_tolerant_trials" # @param ["complete_trials", "fault_tolerant_trials"]
ENTRY_ID = "fault_tolerant_trials_0015" # @param {type:"string"}
launch_annotation_checker(store=store, entry_type=ENTRY_TYPE, entry_id=ENTRY_ID)

### 🎬 Frame-by-Frame Video Player

In [ ]:
# @title 🎬 Frame-by-Frame Video Player (fast) { display-mode: "form" }
from src.annotation import render_video_player

proxy_height = 720 # @param [480, 540, 720, 1080] {type:"raw"}

def _colab_port_url(port: int) -> str:
    try:
        from google.colab import output
        output.no_vertical_scroll()
        return output.eval_js(f"google.colab.kernel.proxyPort({port})") or f"http://localhost:{port}"
    except Exception:
        return f"http://localhost:{port}"

render_video_player(
    store=store,
    entry_type=ENTRY_TYPE,
    entry_id=ENTRY_ID,
    label="synced_video",
    proxy_height=proxy_height,
    port_url_fn=_colab_port_url,
)